# Connecting to the scope

In [ ]:
# This cell is used to show the USB address for the scope. NI-VISA was previously installed
# in the computer for the following to work. It may be found here, https://www.ni.com/en/support/downloads/drivers/download.ni-visa.html

import pyvisa

rm = pyvisa.ResourceManager()
print("Resources:", rm.list_resources())

Following the docs of PyVisa, https://www.pyvisa.com/docs/tektronix-instruments, this is the first cell that they are using

In [ ]:
import pyvisa

rm = pyvisa.ResourceManager()

# USB
scope = rm.open_resource("USB0::0x0699::0x03A6::C047327::INSTR")

scope.timeout = 10000  # 10 s (Tektronix instruments often need longer)
print(scope.query("*IDN?").strip())

scope.write("*RST")
scope.write("*CLS")

# Simple functions which find the scope address and connect to the scope

In [ ]:
from revspace.instrument_drivers.tektronix_scope import obtain_scope_address
serial_number = 'C047327'

scope_address = obtain_scope_address(serial_number)

In [ ]:
from revspace.instrument_drivers.tektronix_scope import connect_to_scope

scope = connect_to_scope(scope_address)

# COMMAND TABLE

## Global commands

In [ ]:
scope.write("ACQuire:STATE ON") # sets the acquisition ON globally (equivalent to pressing "Run")

In [ ]:
scope.write("ACQuire:STATE OFF") # sets the acquisition OFF globally (equivalent to pressing "Stop")

In [ ]:
scope.write("LOCk ALL") # Locks all front-panel controls, useful during measurement!

In [ ]:
scope.write("LOCk NONe") # Unlocks all front-panel controls, useful during measurement!

In [ ]:
scope.write("HORizontal:SCAle 1e-6") # sets the horizontal scale division in units of [s], e.g. 1 us/div

## Channel-specific commands

In [ ]:
scope.write("SELECT:CH1 ON") # turns a channel ON, e.g. Channel 1

In [ ]:
scope.write("SELECT:CH1 OFF") # turns a channel OFF, e.g. Channel 1

In [ ]:
scope.write("CH1:SCAle 0.5") # sets voltage division, in units of [V], e.g. 500 mV/div

In [ ]:
scope.write("CH1:POSition 0.5") # sets the vertical voltage position, in units of [V], e.g. 0.5 V

# Development

In [ ]:
import pyvisa

class Scope:

    def __init__(self,
                 serial_number = 'C047327'):
        
        scope_address = self.obtain_scope_address(serial_number)
        self.com = self.connect_to_scope(scope_address) # self.com is short for self.communications

        self.configuration = {
            'channels': {
                'CH1': {'scale [V]': 1.0,
                        'position [V]': 0.0},
                'CH2': {'scale [V]': 1.0,
                        'position [V]': 0.0},
                'CH3': {'scale [V]': 1.0,
                        'position [V]': 0.0},
                'CH4': {'scale [V]': 1.0,
                        'position [V]': 0.0},
            },
            'global': {'horizontal scale [s/div]': 1e-6}
        }

        # initialization
        self.com.write("ACQuire:STATE ON")
        self.com.write("LOCk NONe")
        self.com.write(f"HORizontal:SCAle {self.configuration['global']['horizontal scale [s/div]']}")
        
        for channel in ['CH1', 'CH2', 'CH3', 'CH4']:
            self.com.write(f"SELECT:{channel} OFF")
            self.com.write(f"{channel}:SCAle {self.configuration['channels'][channel]['scale [V]']}")
            self.com.write(f"{channel}:POSition {self.configuration['channels'][channel]['position [V]']}")
        self.com.write(f"SELECT:CH1 ON") # turn on only CH1


    def obtain_scope_address(self,
                             serial_number):
        """
        Searches all resources and tries to identify an address
        linked to the input serial number.

        Args:
            serial_number (str):
                The input serial number of the device.
                For the Tektronix TDS 2024C it can be found in by pressing
                "Utility > System Status" in the front panel.

        """
        rm = pyvisa.ResourceManager()
        for resource_address in rm.list_resources():
            if serial_number in resource_address:
                print(f"Device with serial number {serial_number} was found with address '{resource_address}'.")
                return resource_address
        raise ValueError(f"Tektronix TDS 2024C with serial number {serial_number} was not found!")
    
    def connect_to_scope(self,
                         scope_address: int):

        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(scope_address)

        scope.timeout = 10000  # 10 s (Tektronix instruments often need longer)
        scope.write("*RST")
        scope.write("*CLS")
        print('Successfully connected:', scope.query("*IDN?").strip())

        return scope

In [ ]:
scope.write("CH1:PRObe 1")

In [ ]:
scope.query("CH1:CURRENTPROBE?")

In [ ]:
scope.write("CH1:INVert OFF")

In [1]:
from revspace.instrument_drivers.tektronix_scope import Scope

scope = Scope()

Device with serial number C047327 was found with address 'USB0::0x0699::0x03A6::C047327::INSTR'.
Successfully connected: TEKTRONIX,TDS 2024C,C047327,CF:91.1CT FV:v24.26


In [ ]:
from typing import Literal

def configure_channel(scope,
                      channel: Literal['CH1', 'CH2', 'CH3', 'CH4'],
                      state: Literal['ON', 'OFF'] = None,
                      coupling: Literal['DC', 'AC', 'GND'] = None,
                      bandwidth: Literal['ON', 'OFF'] = None,
                      probe: Literal[1, 10, 20, 50, 100, 500, 1000] = None,
                      invert: Literal['ON', 'OFF'] = None,
                      scale: float = None,
                      position: float = None):
    
    if state is not None:
        scope.write(f"SELECT:{channel} {state}")
    if coupling is not None:
        scope.write(f"{channel}:COUPling {coupling}")
    if bandwidth is not None:
        scope.write(f"{channel}:BANdwidth {bandwidth}")
    if probe is not None:
        scope.write(f"{channel}:PRObe {probe}")
    if invert is not None:
        scope.write(f"{channel}:INVert {invert}")
    if scale is not None:
        scope.write(f"{channel}:SCAle {scale}")
    if position is not None:
        scope.write(f"{channel}:POSition {position}")

In [8]:
configure_channel(scope.com,
                  channel='CH1',
                  coupling='DC')

In [ ]:
import time
import numpy as np

def single_measurement(scope,
                       channel: str):

    if channel not in ['CH1', 'CH2', 'CH3', 'CH4']:
        raise ValueError("Invalid input channel; valid inputs are: 'CH1', 'CH2', 'CH3', 'CH4'.")

    # Binary encoding for fast transfer
    scope.write("DAT:ENC RIB")
    scope.write("DAT:WID 2")      # 16-bit samples
    scope.write(f"DAT:SOU {channel}")

    # Trigger
    scope.write("TRIG:MAI:TYP EDGE")
    scope.write(f"TRIG:MAI:EDGE:SOU {channel}")
    scope.write("TRIG:MAI:LEV 0.1")
    scope.write("TRIG:MAI:EDGE:SLO RISE")

    # Single acquisition
    scope.write("ACQ:MODE SAM")
    scope.write("ACQ:STOPA SEQ")
    scope.write("ACQ:STATE RUN")

    # Wait for trigger
    start = time.time()
    while time.time() - start < 30:
        if scope.query("ACQ:STATE?").strip() == "0":
            break
        time.sleep(0.1)

    wfmp = scope.query("WFMP?").split(";")

    x_scale = float(wfmp[8])
    y_scale = float(wfmp[12])
    y_zero = float(wfmp[13])
    y_offset  = float(wfmp[14])

    raw_voltage_data = np.array(scope.query_binary_values("CURV?", datatype="h", is_big_endian=True))
    voltages = (raw_voltage_data - y_offset) * y_scale + y_zero
    measurement_times = np.arange(len(raw_voltage_data)) * x_scale - 5e-3

    return {'measurement_times [s]': measurement_times, 'voltages [V]': voltages}

In [ ]:
import pyvisa
import numpy as np
import time

# Binary encoding for fast transfer
scope.write("DAT:ENC RIB")
scope.write("DAT:WID 2")      # 16-bit samples
scope.write("DAT:SOU CH1")

# Horizontal and vertical
scope.write("HOR:SCA 5e-3")   # 1 us/div
scope.write("CH1:SCA 0.5")    # 500 mV/div

# Trigger
scope.write("TRIG:MAI:TYP EDGE")
scope.write("TRIG:MAI:EDGE:SOU CH1")
scope.write("TRIG:MAI:LEV 0.1")
scope.write("TRIG:MAI:EDGE:SLO RISE")

# Single acquisition
scope.write("ACQ:MODE SAM")
scope.write("ACQ:STOPA SEQ")
scope.write("ACQ:STATE RUN")

# Wait for trigger
start = time.time()
while time.time() - start < 30:
    if scope.query("ACQ:STATE?").strip() == "0":
        break
    time.sleep(0.1)

# # Read preamble for scaling
# preamble = scope.query("WFMP?").split(",")
# y_scale = float(preamble[13])
# y_offset = float(preamble[14])
# x_scale = float(preamble[9])

# # Binary transfer
# raw = scope.query_binary_values("CURV?", datatype="h")
# voltages = (np.array(raw) - y_offset) * y_scale
# time_axis = np.arange(len(raw)) * x_scale

# print(f"Captured {len(raw)} points")

In [ ]:
preamble

In [ ]:
y_offset

In [ ]:
wfmp = scope.query("WFMP?").split(";")

x_scale = float(wfmp[8])
y_scale = float(wfmp[12])
y_zero = float(wfmp[13])
y_offset  = float(wfmp[14])

raw = np.array(scope.query_binary_values("CURV?", datatype="h", is_big_endian=True))
voltages = (raw - y_offset) * y_scale + y_zero
time_axis = np.arange(len(raw)) * x_scale - 5e-3

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(dpi=300)

ax.scatter(time_axis, raw)
ax.axhline(0.0, color='black', linewidth = 1, linestyle = '--')

In [ ]:
raw = scope.query_binary_values("CURV?", datatype="h")

In [ ]:
y_scale = float(preamble[2].split(' ')[1])

In [ ]:
preamble[2].split(' ')[1]

In [ ]:
y_offset = float(preamble[14])